# S1 · Clustering by daily balance behaviour

We use each customer's balance history to identify groups with similar activity, balance state, temporal dynamics, regime changes and visibility graph structure. We compare feasible cluster counts up to 12 using separation, stability, group sizes, graph structure and reference simulations. We select the primary partition for the per-client forecasting trials in Stage 2.

We match the sample and cutoff to Stage 0, read its three output tables from `data/prepared/<dataset>/<cutoff>/` and run the analysis in memory.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
import yaml
from IPython.display import display

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/bank_balance_forecasting").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
pio.renderers.default = "plotly_mimetype"
pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(10)


def style_figure(figure, height=420):
    """Fit the output column and keep the plot readable at notebook scale."""
    figure.update_layout(
        width=None,
        autosize=True,
        height=height,
        template="plotly_white",
        font_size=12,
        title_font_size=15,
        margin=dict(l=60, r=28, t=68, b=70),
        legend=dict(orientation="h", x=0, y=-0.19, yanchor="top", font_size=11),
    )
    for menu in figure.layout.updatemenus:
        menu.update(x=1, xanchor="right", y=1.02, yanchor="bottom")
    return figure


def show(figure, height=420):
    """Render an interactive figure at the width of the notebook output."""
    style_figure(figure, height).show(config={"responsive": True, "displaylogo": False})


def choose_partition(figure, title, *, silhouette=False):
    """Show one candidate in the available width while keeping every K selectable."""
    panel_axes = list(dict.fromkeys(trace.xaxis for trace in figure.data))
    if not panel_axes:
        return go.Figure().update_layout(title=title)

    result = go.Figure()
    for trace in figure.data:
        result.add_trace(trace)
        if silhouette:
            result.data[-1].update(marker_line_width=0)
        result.data[-1].update(xaxis="x", yaxis="y", visible=trace.xaxis == panel_axes[0])

    buttons = []
    for axis in panel_axes:
        mask = [trace.xaxis == axis for trace in figure.data]
        label = next(trace.name.split("/")[0] for trace in figure.data if trace.xaxis == axis)
        shapes = []
        if silhouette:
            scores = [score for trace in figure.data if trace.xaxis == axis for score in trace.x]
            mean = sum(scores) / len(scores)
            shapes = [
                dict(
                    type="line",
                    x0=mean,
                    x1=mean,
                    y0=0,
                    y1=1,
                    yref="paper",
                    line=dict(color="#263445", dash="dash", width=2),
                )
            ]
        buttons.append(
            dict(
                label=label,
                method="update",
                args=[{"visible": mask}, {"title.text": f"{title} ({label})", "shapes": shapes}],
            )
        )

    result.update_layout(
        title=f"{title} ({buttons[0]['label']})",
        showlegend=False,
        shapes=buttons[0]["args"][1]["shapes"],
        updatemenus=[dict(buttons=buttons)],
    )
    if silhouette:
        result.update_layout(bargap=0, xaxis_title="Silhouette score", yaxis_title="Accounts")
        result.update_xaxes(range=[-1, 1])
        result.update_yaxes(showticklabels=False)
    else:
        result.update_layout(xaxis_title="Projection axis 1", yaxis_title="Projection axis 2")
        for coordinate in ("x", "y"):
            values = [v for trace in result.data for v in getattr(trace, coordinate)]
            low, high = min(values), max(values)
            padding = max((high - low) * 0.06, 0.01)
            result.update_layout({f"{coordinate}axis_range": [low - padding, high + padding]})
    return result

## 1 · Validation of Stage 0 inputs

We load the prepared balances, calendar profiles and event flags. We check the dataset and cutoff against the provenance record and confirm that balances and events end by that cutoff. These checks establish the information available to the analysis. We use 50 account subsamples for stability and 100 simulations per gap reference distribution.

In [2]:
DATASET = "medium"  # "small" also works
AS_OF_DATE = "2022-12-31"  # use the same values in both notebooks
DATA_ROOT = ROOT / "data"
STAGE0_DIR = DATA_ROOT / "prepared" / DATASET / AS_OF_DATE
settings = yaml.safe_load((ROOT / "parameters.yml").read_text())

parameters = {**settings["stage1"], "as_of_date": AS_OF_DATE}
required = ["prepared_balances", "calendar_profiles", "event_flags"]
missing = [
    name
    for name in [*(f"{name}.parquet" for name in required), "metadata.json"]
    if not (STAGE0_DIR / name).is_file()
]
if missing:
    raise FileNotFoundError(
        f"We could not find Stage 0 outputs for {DATASET=} and {AS_OF_DATE=}. Missing files {missing}"
    )
metadata = json.loads((STAGE0_DIR / "metadata.json").read_text())
assert (metadata["dataset"], metadata["as_of_date"]) == (DATASET, AS_OF_DATE)
prepared_balances = pl.read_parquet(STAGE0_DIR / "prepared_balances.parquet")
calendar_profiles = pl.read_parquet(STAGE0_DIR / "calendar_profiles.parquet")
event_flags = pl.read_parquet(STAGE0_DIR / "event_flags.parquet")
assert str(prepared_balances["date"].max()) <= AS_OF_DATE
assert event_flags.is_empty() or str(event_flags["date"].max()) <= AS_OF_DATE
display(prepared_balances.head())
display(
    pl.DataFrame(
        {
            "dataset": [DATASET],
            "cutoff": [AS_OF_DATE],
            "accounts": [prepared_balances["account_id"].n_unique()],
            "resamples": [parameters["resamples"]],
            "references_per_gap_distribution": [parameters["gap_references"]],
        }
    )
)

dataset,cutoff,accounts,resamples,references_per_gap_distribution
str,str,i64,i64,i64
"""medium""","""2022-12-31""",500,50,100


In [3]:
from bank_balance_forecasting.clustering.clustering import (
    assess_stability,
    assign_accounts,
    fit_candidates,
)
from bank_balance_forecasting.clustering.diagnostics import (
    assess_neighborhood_sensitivity,
    compare_feature_sets,
)
from bank_balance_forecasting.clustering.evaluation import (
    assess_cluster_count,
    estimate_gap,
    evaluate_separation,
    inspect_spectrum,
)
from bank_balance_forecasting.clustering.evidence_plots import plot_count_tests
from bank_balance_forecasting.clustering.features import (
    extract_activity,
    extract_balance_state,
    extract_dynamics,
    extract_regime_features,
    feature_label,
)
from bank_balance_forecasting.clustering.graphs import (
    extract_hvg_features,
    extract_nvg_features,
)
from bank_balance_forecasting.clustering.plots import (
    explain_groups,
    plot_ablation,
    plot_clusters,
    plot_selection,
)
from bank_balance_forecasting.clustering.representation import build_representation
from bank_balance_forecasting.clustering.spectral import build_customer_similarity_graph

## 2 · History length and activity

We calculate observed-history length, movement frequency, flat runs and spacing between movements for each account. We use these features to distinguish short histories from long periods of observed inactivity and to identify differences in account usage. The preview shows one row per account.

In [4]:
activity_features = extract_activity(prepared_balances.lazy(), parameters)
display(activity_features.select(activity_features.columns[:8]).head())

## 3 · Balance state and variability

We summarize the level and spread of balances, the negative-balance share, sign crossings and movement variability. These robust features complement the activity and temporal descriptors. The preview shows the characteristics we use in scaling and distance calculations.

In [5]:
state_features = extract_balance_state(prepared_balances.lazy(), parameters)
display(state_features.select(state_features.columns[:8]).head())

## 4 · Temporal dependence and calendar regularity

We use selected tsfresh calculators for change autocorrelation, spectral entropy and complexity. We measure weekday activity and inflow regularity to describe timing. These measurements distinguish accounts with similar balances and movement frequencies whose movements follow different temporal patterns. We preserve daily timing and mark descriptors unavailable when the data requirements are unmet.

In [6]:
dynamics_features = extract_dynamics(prepared_balances.lazy(), calendar_profiles.lazy(), parameters)
display(dynamics_features.select(dynamics_features.columns[:8]).head())

## 5 · Changes in account behaviour

We compare recent levels and activity with the full history and summarize monthly variation and Stage 0 events. This helps us identify shifts that full-history averages can hide. For recent movement scale, we cap absolute changes within the descriptor calculation because many zero changes can cause median absolute deviation to collapse.

In [7]:
regime_features = extract_regime_features(prepared_balances.lazy(), event_flags.lazy(), parameters)
display(regime_features.select(regime_features.columns[:8]).head())

## 6 · Shape descriptors from natural visibility graphs

A natural visibility graph links observations when every intermediate observation lies below the straight line joining them. We use graph degree, transitivity, entropy, edge span and directed asymmetry to summarize how peaks and movements connect across time. We compute the descriptors on balance levels and changes in complete 90-day windows with a 30-day stride and a final window ending at the last observation. We summarize typical structure and variability with medians and interquartile ranges and report window counts for coverage. We test the effect on the partitions in Section 16.

In [8]:
nvg_features = extract_nvg_features(prepared_balances.lazy(), parameters)
display(nvg_features.select(nvg_features.columns[:8]).head())

## 7 · Shape descriptors from horizontal visibility graphs

A horizontal visibility graph links two observations when every intermediate observation is below both endpoints. It captures the ordering of local peaks. We use the same windows and summaries as for natural visibility graphs to compare coverage and features consistently. In Section 16, we measure how each graph family affects membership and stability.

In [9]:
hvg_features = extract_hvg_features(prepared_balances.lazy(), parameters)
display(hvg_features.select(hvg_features.columns[:8]).head())

## 8 · Comparable dissimilarities between customers

We remove descriptors with insufficient coverage, constant values or strong redundancy across families. We scale retained descriptors to [0, 1] using fitted 5th and 95th percentiles. We assign 20% weight to each statistical family and 10% to each visibility graph family. We average within families to limit the influence of families with many descriptors, then renormalize the weights over comparable descriptors for each pair.

We require at least 90 observed days and sufficient family coverage for clustering. We report removals and eligibility to show which accounts and descriptors enter the graph.

In [10]:
feature_values, feature_matrix, feature_registry, distances = build_representation(
    activity_features.lazy(),
    state_features.lazy(),
    dynamics_features.lazy(),
    regime_features.lazy(),
    nvg_features.lazy(),
    hvg_features.lazy(),
    parameters,
)
display(
    pl.DataFrame(
        {
            "accounts": [feature_matrix.height],
            "eligible": [feature_matrix["eligible"].sum()],
            "retained_features": [len(feature_registry["features"])],
            "removed_features": [len(feature_registry["removed"])],
        }
    )
)
removed = pl.DataFrame(
    {
        "feature": list(feature_registry["removed"]),
        "reason": list(feature_registry["removed"].values()),
    }
)
display(removed.group_by("reason").len() if removed.height else removed)
if feature_matrix.filter("eligible").height < 3:
    raise ValueError(
        "Too few eligible accounts for a clustering study. Inspect history/coverage above."
    )

accounts,eligible,retained_features,removed_features
i64,i64,i64,i64
500,500,76,8


reason,len
str,u32
"""constant""",1
"""redundant_across_families""",7


### Feature coverage

We count accounts with at least one unavailable descriptor in each family. The bars show the share of all accounts affected when gaps occur. We use these counts to identify families with uneven coverage, and the eligibility summary records how many accounts enter clustering.

In [11]:
features = [name for name in feature_values.columns if "__" in name]
coverage_rows = []
for family in sorted({name.split("__")[0] for name in features}):
    columns = [name for name in features if name.startswith(f"{family}__")]
    finite = np.isfinite(feature_values.select(columns).to_numpy().astype(float))
    missing = int((~finite.all(axis=1)).sum())
    coverage_rows.append((family, missing, len(finite)))
coverage_rows.sort(key=lambda row: (row[1], row[0]))
family, missing, totals = zip(*coverage_rows)
if not any(missing):
    print(f"All {totals[0]} accounts have complete coverage across {len(family)} feature families.")
else:
    coverage_plot = go.Figure(
        go.Bar(
            x=[count / total for count, total in zip(missing, totals)],
            y=family,
            orientation="h",
            text=[f"{count}/{total}" for count, total in zip(missing, totals)],
            textposition="outside",
            marker_color="#357a8a",
            cliponaxis=False,
            hovertemplate="%{y}: %{text} accounts<extra></extra>",
        )
    )
    coverage_plot.update_layout(title="Accounts with missing features by family", showlegend=False)
    coverage_plot.update_xaxes(
        range=[0, 1.14],
        tickformat=".0%",
        tickvals=[0, 0.25, 0.5, 0.75, 1],
        title_text="Share with at least one missing feature",
    )
    show(coverage_plot, 360)

All 500 accounts have complete coverage across 6 feature families.


### Associations among retained features

We rank the eight strongest remaining pairs by absolute Spearman correlation. We report the sign to show the direction of association. The ranking helps us identify descriptors that may capture related behaviour after pruning.

In [12]:
correlations = (
    feature_matrix.select(feature_registry["features"]).to_pandas().corr(method="spearman")
)
strongest = []
for i, left in enumerate(correlations.columns):
    for j, right in enumerate(correlations.columns[:i]):
        rho = correlations.iat[i, j]
        if np.isfinite(rho):
            strongest.append((feature_label(left), feature_label(right), float(rho)))
strongest.sort(key=lambda pair: abs(pair[2]), reverse=True)
display(pl.DataFrame(strongest[:8], schema=["Feature 1", "Feature 2", "Spearman ρ"], orient="row"))

Feature 1,Feature 2,Spearman ρ
str,str,f64
"""Frequency of crossing zero""","""Time with a negative balance""",0.899717
"""hvg: changes max degree share median""","""nvg: changes degree entropy median""",0.893187
"""hvg: changes mean degree iqr""","""hvg: changes degree entropy iqr""",0.89211
"""Movement variability relative to balances""","""Balance variability""",-0.890095
"""Typical balance level""","""Time with a negative balance""",-0.881613
"""hvg: levels time asymmetry median""","""hvg: levels max degree share median""",0.869332
"""nvg: changes temporal span iqr""","""nvg: changes max degree share iqr""",0.861408
"""hvg: levels transitivity median""","""nvg: changes degree entropy median""",0.860199


## 9 · Spectral clustering on the customer similarity graph

We build a symmetric nearest-neighbor graph from the balanced customer dissimilarities. We use local affinity scaling to account for differences in neighborhood density. Spectral clustering identifies communities connected by strong similarities and can capture irregular group shapes. We fit every feasible K up to 12 and include K=1 as the unsegmented control. We report which candidates the graph supports and flag groups that may be too small for forecasting comparisons.

In [13]:
similarity_graph = build_customer_similarity_graph(distances.lazy(), parameters)
candidate_models, candidate_metrics = fit_candidates(
    distances.lazy(),
    similarity_graph.lazy(),
    parameters,
)
display(candidate_metrics)

k,loss,silhouette,minimum_group_size,size_ok,graph_components,neighbors
i64,f64,f64,i64,bool,i64,i64
1,121.217092,null,500,true,1,25
2,110.080094,0.151493,177,true,1,25
3,102.166886,0.160497,151,true,1,25
4,98.589927,0.141993,79,true,1,25
…,…,…,…,…,…,…
9,92.103501,0.098292,36,true,1,25
10,92.07656,0.088437,24,true,1,25
11,89.977446,0.09257,24,true,1,25
12,88.423222,0.087851,20,true,1,25


## 10 · Stability under account subsampling

We refit pruning, scaling and graph construction on the same 50 random 80% subsamples for each K. We compare the refitted and full-data partitions on shared accounts. We use adjusted Rand index (ARI) for overall agreement and matched Jaccard overlap for individual groups, including small groups whose changes can have little effect on ARI. This tests reproducibility when the customer sample changes.

In [14]:
stability_metrics, assignment_stability, stability_trials = assess_stability(
    feature_values.lazy(),
    feature_matrix.lazy(),
    candidate_models,
    parameters,
)
display(stability_metrics)

k,mean_ari,std_ari,ari_q10,ari_q90,minimum_mean_jaccard
i64,f64,f64,f64,f64,f64
2,0.818723,0.148074,0.656146,0.941458,0.882457
3,0.858524,0.071492,0.763658,0.950143,0.874269
4,0.86392,0.071325,0.752542,0.940016,0.858476
5,0.771613,0.103439,0.657306,0.872959,0.774896
…,…,…,…,…,…
9,0.684189,0.080062,0.557322,0.774863,0.475129
10,0.505302,0.080428,0.416746,0.619207,0.414243
11,0.497732,0.070236,0.409838,0.59165,0.371488
12,0.562611,0.088088,0.45026,0.670699,0.433793


## 11 · Separation and graph structure

We compute silhouette and Dunn using the balanced dissimilarities that define the fit. We use a weighted Euclidean feature view for Calinski-Harabasz and Davies-Bouldin to obtain complementary measures of compactness and separation. We use the normalized-Laplacian spectrum to test for community structure across neighborhood sizes. We report the available scores for each candidate and examine their agreement in the figures below.

In [15]:
quality_metrics, silhouette_samples = evaluate_separation(
    distances.lazy(),
    feature_matrix.lazy(),
    feature_registry,
    candidate_models,
)
spectrum_metrics = inspect_spectrum(distances.lazy(), parameters)
display(quality_metrics)

k,within_dispersion,silhouette,negative_silhouette_share,dunn,zero_within_diameter,calinski_harabasz_euclidean,davies_bouldin_euclidean,euclidean_view_available
i64,f64,f64,f64,f64,bool,f64,f64,bool
1,23.757336,null,null,null,false,null,null,false
2,20.12056,0.151493,0.088,0.185005,false,72.30763,2.428526,true
3,16.675117,0.160497,0.106,0.223202,false,82.678785,2.00687,true
4,15.39205,0.141993,0.082,0.186807,false,67.695948,2.118352,true
…,…,…,…,…,…,…,…,…
9,12.717682,0.098292,0.158,0.172018,false,40.67973,2.297877,true
10,12.679033,0.088437,0.212,0.172018,false,36.689114,2.228275,true
11,12.066715,0.09257,0.182,0.207404,false,35.583965,2.281419,true
12,11.717347,0.087851,0.162,0.218517,false,34.109797,2.423819,true


## 12 · Compactness against simulated references

We compute Gap(K) as the mean simulated log dispersion minus the observed log dispersion. We fit clustering to 100 draws from an axis-aligned uniform box and 100 from a PCA-aligned uniform box, holding the fitted feature representation fixed. This estimates how much tighter the observed groups are than groups obtained under the two reference assumptions. We report valid draw counts and simulation uncertainty.

In [16]:
gap_metrics, gap_reference_trials = estimate_gap(
    feature_matrix.lazy(),
    feature_registry,
    candidate_models,
    parameters,
)
display(gap_metrics)

reference,k,log_w,reference_log_w,gap,gap_se,mc_se_mean,valid_references,requested_references
str,i64,f64,f64,f64,f64,f64,i64,i64
"""axis_uniform""",1,3.167891,3.334822,0.16693,0.006558,0.000652,100,100
"""axis_uniform""",2,3.001742,3.287179,0.285437,0.006901,0.000687,100,100
"""axis_uniform""",3,2.813918,3.2559,0.441982,0.007531,0.000749,100,100
"""axis_uniform""",4,2.733851,3.238355,0.504504,0.01009,0.001004,100,100
…,…,…,…,…,…,…,…,…
"""pca_uniform""",9,2.542993,3.974403,1.43141,0.018314,0.001822,100,100
"""pca_uniform""",10,2.53995,3.954592,1.414643,0.015283,0.001521,100,100
"""pca_uniform""",11,2.490451,3.93738,1.446929,0.014369,0.00143,100,100
"""pca_uniform""",12,2.46107,3.924203,1.463133,0.015494,0.001542,100,100


## 13 · Evidence across cluster counts

We use separation to assess fit, resampling and sensitivity to examine reproducibility, and simulated references to assess compactness. We also check group sizes for later forecasting trials. The figures use the same candidate partitions, so related scores can corroborate one another.

In [17]:
k_assessment, selection_metrics = assess_cluster_count(
    candidate_metrics.lazy(),
    stability_metrics.lazy(),
    quality_metrics.lazy(),
    spectrum_metrics.lazy(),
    gap_metrics.lazy(),
    parameters,
)
selection_plot = plot_selection(selection_metrics.lazy(), k_assessment, parameters)
quality_plot, silhouette_plot, spectrum_plot, gap_plot, dispersion_plot, _ = plot_count_tests(
    quality_metrics.lazy(),
    silhouette_samples.lazy(),
    spectrum_metrics.lazy(),
    gap_metrics.lazy(),
    selection_metrics.lazy(),
)

### Separation, reproducibility and group size

We examine mean silhouette, mean resample ARI, the weakest group's mean matched Jaccard and the smallest group size together. These panels show which partitions combine separation, repeatable memberships and adequate group sizes. Dotted lines mark the stability thresholds. Shading covers the empirical 10th to 90th ARI percentiles across subsamples. Separation and stability scores begin at K=2.

In [18]:
selection_plot.update_layout(title="Separation, stability and group size")
show(selection_plot, 680)

### Complementary separation scores

| Measure | What it measures | Why it helps |
|---|---|---|
| Dunn index | Minimum between-group distance divided by maximum within-group distance | Tests separation in the worst case, with sensitivity to individual pairs |
| Negative silhouette share | Fraction of customers closer on average to another group | Quantifies ambiguous placements |
| Calinski-Harabasz | Between-group relative to within-group variation in weighted Euclidean space | Higher values favor compact, separated groups |
| Davies-Bouldin | Average similarity to each group's closest competitor in weighted Euclidean space | Lower values favor less overlap with neighboring groups |

The two Euclidean scores require complete inputs in that feature view. We compare the patterns across scores to see whether separation depends on the definition used.

In [19]:
quality_plot.update_layout(title="Complementary separation scores")
show(quality_plot, 670)

### Individual silhouette distributions

We use the menu to display the silhouette distribution for each K. Each bar represents one customer and is ordered within its group. Positive values indicate smaller average dissimilarity within the assigned group. Near-zero values indicate a boundary, and negative values indicate a closer competing group. The dashed line shows the mean. We examine the distributions for ambiguous memberships that a high overall mean can hide.

In [20]:
show(choose_partition(silhouette_plot, "Silhouette distribution", silhouette=True), 465)

### Eigenvalues and eigengaps

We plot the small normalized-Laplacian eigenvalues above and the differences between eigenvalues K and K+1 below. A large gap supports K graph communities. We compare curves across neighborhood sizes to assess whether the same count recurs under different graph settings. K=1 represents evidence for a single community.

In [21]:
spectrum_plot.update_layout(title="Graph eigenvalues and eigengaps")
show(spectrum_plot, 600)

### Gap statistic by reference distribution

We compare observed compactness with one uniform-box reference per curve. Error bars show the gap rule's s(K). The legend gives the first K for which Gap(K) is at least Gap(K+1) minus s(K+1). We use agreement between the curves to assess sensitivity to the two references. The hover labels report valid simulation counts and Monte Carlo error.

In [22]:
gap_plot.update_layout(title="Gap statistic by reference distribution")
show(gap_plot, 420)

### Observed and reference dispersion

We plot within-group squared-dissimilarity dispersion in the upper panel. A flattening curve shows where additional groups yield smaller gains in compactness. We fit each spectral partition independently, so increases can occur. The lower panel shows the observed and simulated log dispersion whose difference gives the gap statistic.

In [23]:
dispersion_plot.update_layout(title="Observed and reference dispersion")
show(dispersion_plot, 590)

## 14 · Customer memberships for each candidate

We record each customer's group and review status for every candidate K. We assign customers with insufficient history or feature coverage to a review group. The table lets us trace memberships and carry eligible groups into forecasting trials.

In [24]:
candidate_client_groups, cluster_model = assign_accounts(
    feature_values.lazy(),
    feature_matrix.lazy(),
    feature_registry,
    candidate_models,
    assignment_stability.lazy(),
    stability_metrics.lazy(),
    distances.lazy(),
    parameters,
)
cluster_plot, _, group_sizes_plot, _ = plot_clusters(
    distances.lazy(),
    candidate_client_groups.lazy(),
)
display(candidate_client_groups.head())

### Candidate memberships in a shared projection

We use the menu to compare candidate partitions on the same approximate two-dimensional projection of the balanced dissimilarities. Fixed axes make membership changes visible. Colors identify groups within each fit. We examine boundaries and overlap in this figure and use full-dimensional diagnostics to compare separation.

In [25]:
show(choose_partition(cluster_plot, "Customer groups in the same projection"), 480)

### Group sizes by candidate K

We plot the customer counts in each group, including the review group when present. We compare the distributions to assess balance, identify small groups and judge how much evidence each group can contribute to later forecasting comparisons.

In [26]:
group_sizes_plot.update_layout(title="Group sizes by candidate K")
show(group_sizes_plot, 425)

## 15 · Interpretation of the groups

We summarize membership, movement frequency, negative-balance share and review status. We compare those summaries with representative histories to relate the groups to observed customer behaviour.

In [27]:
group_profiles, summary, group_profiles_plot, representative_histories_plot = explain_groups(
    feature_values.lazy(),
    feature_matrix.lazy(),
    candidate_client_groups.lazy(),
    prepared_balances.lazy(),
    k_assessment,
    feature_registry,
    parameters,
)
display(group_profiles.head(12))

### Group feature profiles

We compare group medians for each candidate K using the figure menu. Colors show values from 0 to 1 on the fitted feature scale. We use differences to assess labels such as higher activity or frequent negative balances. Similar profiles indicate limited differentiation on the descriptors shown.

In [28]:
group_profiles_plot.update_layout(title="Feature profiles by candidate K")
show(group_profiles_plot, 550)

### Representative balance histories

We use the figure menu to examine a representative account's full history for each group. We center each history by its median and scale it by its interquartile range so the shapes are comparable. The traces illustrate activity, flat periods, level shifts and excursions. We use silhouette distributions and review flags to judge how broadly each example applies.

In [29]:
representative_histories_plot.update_layout(title="Representative histories by candidate K")
show(representative_histories_plot, 470)

## 16 · Sensitivity to visibility features and graph settings

We refit four representations using statistics alone, statistics with natural visibility, statistics with horizontal visibility, and all features. We use ARI in the top panel to measure changes in membership relative to the combined representation. The middle panel compares resampling stability, and the bottom panel compares memberships across graph neighborhood sizes. These checks show which conclusions depend on specific feature or graph choices.

In [30]:
ablation_metrics = compare_feature_sets(
    feature_values.lazy(), feature_matrix.lazy(), candidate_models, parameters
)
neighborhood_metrics = assess_neighborhood_sensitivity(
    distances.lazy(), candidate_models, parameters
)
show(
    plot_ablation(ablation_metrics.lazy(), neighborhood_metrics.lazy()).update_layout(
        title="Sensitivity to features and graph settings"
    ),
    690,
)

## 17 · Summary of diagnostic results

We report each diagnostic's preferred count and use the stability and size screen to identify candidates with adequate agreement and group size. We verify that every input account appears in the assignment table.

In [31]:
display(
    pl.DataFrame(
        {
            "diagnostic": list(k_assessment["diagnostic_suggestions"]),
            "suggestion": [
                json.dumps(value) for value in k_assessment["diagnostic_suggestions"].values()
            ],
        }
    )
)
print("Candidates passing the stability/size screen:", k_assessment["stable_candidates"])
assert (
    candidate_client_groups["account_id"].n_unique() == prepared_balances["account_id"].n_unique()
)

diagnostic,suggestion
str,str
"""silhouette""","""3"""
"""dunn""","""3"""
"""calinski_harabasz_euclidean""","""3"""
"""davies_bouldin_euclidean""","""3"""
"""eigengap_by_neighbors""","""{""10"": 3, ""15"": 3, ""25"": 3, ""4…"
"""gap_1se_by_reference""","""{""axis_uniform"": 6, ""pca_unifo…"


Candidates passing the stability/size screen: [2, 3, 4, 5, 6]


## Conclusion

We selected **three clusters (K=3)** for the 500-account sample through 31 December 2022. K=3 has the highest mean silhouette (0.160), Dunn index and Calinski-Harabasz score, the lowest Davies-Bouldin score, and the largest eigengap at all five neighborhood sizes. Its mean resampling ARI is 0.859, close to 0.864 for K=4, and the groups contain 151, 163 and 186 accounts. Both gap references favor K=6, reflecting finer structure under their compactness criterion. Agreement across separation measures and graph settings, strong stability and balanced group sizes support that choice. In Stage 2, we compare it with the K=1 control using chronological 30-day per-client forecast errors.